In [2]:
import h5py
import numpy as np
import torch
import os
from tqdm.notebook import tqdm  # 如果在终端运行，请改为 from tqdm import tqdm
from pathlib import Path
from scipy import ndimage as ndi
import datetime as dt

from RAFT.core.raft import RAFT

class DotDict(dict):
    def __getattr__(self, key):
        return self[key]
    def __setattr__(self, key, value):
        self[key] = value

ROOT = Path.cwd()
if not (ROOT / "RAFT").exists() and (ROOT.parent / "RAFT").exists():
    ROOT = ROOT.parent

WEIGHTS = ROOT / "RAFT" / "models" / "raft-small.pth"

# 降噪配置
DENOISE_CFG = {
    "median_size": 3,
    "tau_abs": 0.6,
    "tau_rel": 0.08,
    "min_component_area": 64,
}

# ==========================================
# 1. 核心模型与算法函数 (保持你的高水准原样)
# ==========================================
def load_raft_model(weight_path=WEIGHTS, small=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    args = DotDict(small=small, mixed_precision=True, alternate_corr=False)
    model = RAFT(args).to(device).eval()

    if not weight_path.exists(): raise FileNotFoundError(f"找不到 RAFT 权重: {weight_path}")

    try: state = torch.load(weight_path, map_location=device, weights_only=True)
    except TypeError: state = torch.load(weight_path, map_location=device)

    if isinstance(state, dict) and "state_dict" in state: state = state["state_dict"]
    if isinstance(state, dict):
        first_key = next(iter(state))
        if first_key.startswith("module."):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state)
    return model, device

def hard_denoise_flow(flow_np, valid_mask, cfg=DENOISE_CFG):
    u, v = flow_np[..., 0], flow_np[..., 1]
    u_med, v_med = ndi.median_filter(u, size=cfg["median_size"]), ndi.median_filter(v, size=cfg["median_size"])
    mag = np.sqrt(u_med ** 2 + v_med ** 2)

    max_mag = float(mag[valid_mask].max()) if valid_mask.any() else float(mag.max())
    tau = max(cfg["tau_abs"], cfg["tau_rel"] * max_mag)

    keep = valid_mask & (mag >= tau)
    labeled, num = ndi.label(keep)
    if num > 0:
        counts = np.bincount(labeled.ravel())
        small_labels = np.where(counts < cfg["min_component_area"])[0]
        if len(small_labels) > 0: keep[np.isin(labeled, small_labels)] = False

    out = np.zeros_like(flow_np, dtype=np.float32)
    out[..., 0][keep] = u_med[keep]
    out[..., 1][keep] = v_med[keep]
    return out, keep, tau

def build_valid_sky_mask(image1_np, image2_np, radius_ratio=0.49, black_threshold=8):
    h, w = image1_np.shape[:2]
    cy, cx = h // 2, w // 2
    r = int(min(h, w) * radius_ratio)
    yy, xx = np.ogrid[:h, :w]
    circle_mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2

    valid1 = np.any(image1_np > black_threshold, axis=2)
    valid2 = np.any(image2_np > black_threshold, axis=2)
    return circle_mask & valid1 & valid2

# ==========================================
# 2. V3 终极光流生成引擎 (单池子 + 读写双缓冲)
# ==========================================
def generate_flow_h5_v3(source_h5, times_dir, output_h5):
    print("⏳ 正在挂载 RAFT 模型...")
    model, device = load_raft_model(small=True)
    
    # 🌟 1. 无缝拼接全量时间轴
    t_train = np.load(os.path.join(times_dir, 'times_trainval.npy'), allow_pickle=True)
    t_test = np.load(os.path.join(times_dir, 'times_test.npy'), allow_pickle=True)
    all_times = np.concatenate([t_train, t_test])
    total_frames = len(all_times)
    print(f"📅 成功加载时间轴，共计 {total_frames} 帧连续数据。")

    with h5py.File(source_h5, 'r') as src, h5py.File(output_h5, 'w') as dst:
        
        # 🌟 2. 单一数据源，杜绝冗余逻辑
        src_imgs = src['trainval']['global_images_log']
        assert src_imgs.shape[0] == total_frames, f"🚨 图像数({src_imgs.shape[0]})与时间戳数({total_frames})不匹配！"

        # 🌟 3. 预分配终极光流池
        dst_grp = dst.create_group('trainval')
        flow_ds = dst_grp.create_dataset(
            'global_flow_log', 
            shape=(total_frames, 2, 256, 256), 
            dtype='float16', 
            chunks=(1, 2, 256, 256), 
            compression='lzf'
        )
        print(f"✅ 已在硬盘上预分配 {total_frames} 帧光流存储空间。")
        
        # 强行塞入第 0 帧占位符 (因为光流是向后差分)
        flow_ds[0] = np.zeros((2, 256, 256), dtype=np.float16)

        # 🌟 4. 双向大巴车：批量读 + 批量写
        buffer_size = 500 # 500 帧一批次，保证内存与显存的绝佳平衡
        
        for start_idx in tqdm(range(1, total_frames, buffer_size), desc="🚀 RAFT 光流引擎全速推进中"):
            end_idx = min(start_idx + buffer_size, total_frames)
            
            # [批量读取]：一次性连同上一帧（start_idx - 1）一起吸入内存
            img_buffer = src_imgs[start_idx - 1 : end_idx]
            
            # 建立一个临时列表，收集这批光流结果
            out_flows = []
            
            for local_i in range(end_idx - start_idx):
                global_i = start_idx + local_i
                
                # 严格的时间断层校验
                time_delta = (all_times[global_i] - all_times[global_i-1]).total_seconds()
                if time_delta > 65:
                    out_flows.append(np.zeros((2, 256, 256), dtype=np.float16))
                    continue
                    
                # 内存级极速取图
                img1 = img_buffer[local_i]
                img2 = img_buffer[local_i + 1]
                
                # Tensor 化推入 GPU
                t1 = torch.from_numpy(img1).float().unsqueeze(0).to(device)
                t2 = torch.from_numpy(img2).float().unsqueeze(0).to(device)
                
                with torch.no_grad():
                    _, flow_up = model(t1, t2, iters=20, test_mode=True)
                    flow_np = flow_up[0].permute(1, 2, 0).cpu().numpy()
                
                # 物理掩码与数学形态学降噪
                img1_np = img1.transpose(1, 2, 0)
                img2_np = img2.transpose(1, 2, 0)
                valid_mask = build_valid_sky_mask(img1_np, img2_np)
                
                flow_dn_raw, _, _ = hard_denoise_flow(flow_np, valid_mask)
                flow_dn = flow_dn_raw.transpose(2, 0, 1).astype(np.float16)
                
                out_flows.append(flow_dn)
                
            # 💥 [批量写入]：把算好的几百帧光流“啪”地一下直接烙在硬盘上
            flow_ds[start_idx : end_idx] = np.stack(out_flows)

    print("\n🎉 宗师级全量光流数据 V3 生成完毕！行号与主数据集 100% 绝对物理对齐！")

# ==========================================
# 3. 启动指令
# ==========================================
if __name__ == '__main__':
    data_dir = r"E:\造数据集\output_folder_v3"
    src_h5 = os.path.join(data_dir, '2019_dataset_dual_V3.h5')
    out_h5 = os.path.join(data_dir, '2019_dataset_flow_V3.h5')
    
    generate_flow_h5_v3(src_h5, data_dir, out_h5)

⏳ 正在挂载 RAFT 模型...
📅 成功加载时间轴，共计 101368 帧连续数据。
✅ 已在硬盘上预分配 101368 帧光流存储空间。


🚀 RAFT 光流引擎全速推进中:   0%|          | 0/203 [00:00<?, ?it/s]

e:\造数据集\RAFT\core\raft.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
e:\造数据集\RAFT\core\raft.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
e:\DL_Tools\anacoda\envs\skippd_torch\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
e:\造数据集\RAFT\core\raft.py:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precis


🎉 宗师级全量光流数据 V3 生成完毕！行号与主数据集 100% 绝对物理对齐！
